In [60]:
import numpy as np
import pandas as pd

from scipy.sparse import hstack
from category_encoders import BinaryEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

In [61]:
# Global Variables
FILE_PATH: str = "https://raw.githubusercontent.com/vinayak-ensemble/Dataset-TV-Shows-OTT/refs/heads/main/tv-shows.csv" 
OTT_DF: pd.DataFrame = pd.read_csv(FILE_PATH)
# OTT_DF.set_index('id', inplace=True)

In [64]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MultiLabelBinarizer

# 1. Create the destination directory if it does not exist
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)

df = OTT_DF.copy()
y_raw = (
    df["listed_in"]
    .astype(str)
    .str.split(",")
    .apply(lambda x: [i.strip() for i in x])
)

mlb = MultiLabelBinarizer()
Y_binarized = mlb.fit_transform(y_raw)

stratification_tags = ["_".join(map(str, row)) for row in Y_binarized]

series_counts = pd.Series(stratification_tags).value_counts()
safe_tags = pd.Series(stratification_tags).map(
    lambda val: val if series_counts[val] > 1 else "rare_combination_group"
).values

train_df, test_df = train_test_split(
    df,
    test_size=0.20,
    stratify=safe_tags,
    random_state=42
)

if "listed_in" in test_df.columns:
    test_df = test_df.drop(columns=["listed_in"])

train_path = os.path.join(output_dir, "train.csv")
test_path = os.path.join(output_dir, "test.csv")

train_df.to_csv(train_path, index=False)
test_df.to_csv(test_path, index=False)

print(f"Stratified data splitting complete.")
print(f"Saved train dataset to: {train_path} (Shape: {train_df.shape})")
print(f"Saved test dataset to: {test_path} (Shape: {test_df.shape})")

Stratified data splitting complete.
Saved train dataset to: data\train.csv (Shape: (7470, 13))
Saved test dataset to: data\test.csv (Shape: (1868, 12))


## 1. Pre-processing

In [3]:
# Converting the date_added to datetime format:
OTT_DF['date_added'] = pd.to_datetime(OTT_DF['date_added'].str.strip(), format='%B %d, %Y')

In [4]:
OTT_DF.columns

Index(['type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description',
       'platform'],
      dtype='str')

In [5]:
# Handle missing values:

# -- High missing value columns --

# Treatment: 
# - need to fill in `Unknown` for the missing values in these cases.
# - since director has a 30% missing value we need to check if it's fit to be modeled on. 

treatment_cols = ['director', 'cast', 'country']
OTT_DF[treatment_cols] = OTT_DF[treatment_cols].fillna('Unknown')
OTT_DF[treatment_cols].isnull().sum()

director    0
cast        0
country     0
dtype: int64

In [6]:
# Fixing the duration column by brining in values from rating
mask = (
    OTT_DF['duration'].isna() &
    OTT_DF['rating'].astype(str).str.contains(r'^\d+\s*min$', na=False)
)

OTT_DF.loc[mask, 'duration'] = OTT_DF.loc[mask, 'rating']
OTT_DF.loc[mask, 'rating'] = np.nan

In [7]:
# Handle missing values:

# -- Low missing value columns --

# Treatment: 
# - need to fill in with the mean of the numeric columns.
# - need to fill in with the mode of the string columns. 

mean_treat_cols = ['date_added' ,'release_year']
mode_tread_grp_cols = ['rating']

# fillna with mean of the columns
for col in mean_treat_cols:
    OTT_DF[col] = OTT_DF[col].fillna(OTT_DF[col].mean())

# fillna with the mode of the column by type
duration_mode_by_type = (
    OTT_DF.groupby('type')[mode_tread_grp_cols[0]]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)

OTT_DF[mode_tread_grp_cols[0]] = OTT_DF[mode_tread_grp_cols[0]].fillna(
    OTT_DF['type'].map(duration_mode_by_type)
)

OTT_DF[mean_treat_cols + mode_tread_grp_cols].isnull().sum()

date_added      0
release_year    0
rating          0
dtype: int64

## 2. Data Transformation

In [8]:
# Transform the duration column into 2 seperate columns based on type.

# extract numeric value
OTT_DF['duration_num'] = OTT_DF['duration'].str.extract(r'(\d+)').astype(float)

# create separate features
OTT_DF['duration_movies'] = np.where(
    OTT_DF['type'] == 'Movie',
    OTT_DF['duration_num'],
    np.nan
)

OTT_DF['duration_tv_series'] = np.where(
    OTT_DF['type'] == 'TV Show',
    OTT_DF['duration_num'],
    np.nan
)

# optional: fill missing values with 0
OTT_DF[['duration_movies', 'duration_tv_series']] = (
    OTT_DF[['duration_movies', 'duration_tv_series']]
    .fillna(0)
)

# drop old columns
OTT_DF.drop(columns=['duration', 'duration_num'], inplace=True)
OTT_DF[['duration_movies', 'duration_tv_series']].head()

,duration_movies,duration_tv_series
id,,
1,61.0,0.0
2,123.0,0.0
3,0.0,1.0
4,0.0,1.0
5,63.0,0.0


In [9]:
# Transform the data to create meaningful bins

cols_to_bin = ['duration_movies', 'duration_tv_series', 'date_added', 'release_year']

# updating the column to only have the year so that we can bin it
OTT_DF['date_added'] = OTT_DF['date_added'].dt.year

# using quartile bins to have more or less equal number of values in each bin
for col in cols_to_bin:
    OTT_DF[f'{col}_bin'] = pd.qcut(
        OTT_DF[col],
        q=6,
        labels=False,
        duplicates='drop'
    )

OTT_DF.drop(
    columns=cols_to_bin,
    inplace=True
)

In [10]:
for col in cols_to_bin:
    print(OTT_DF[f'{col}_bin'].value_counts(), end='\n\n')

duration_movies_bin
0    3120
2    1639
1    1583
4    1548
3    1448
Name: count, dtype: int64

duration_tv_series_bin
0    8357
1     981
Name: count, dtype: int64

date_added_bin
1    4055
2    2030
3    1678
0    1575
Name: count, dtype: int64

release_year_bin
2    2414
4    1997
0    1678
1    1498
3    1086
5     665
Name: count, dtype: int64



In [11]:
# binary encodding the rating column as having 17 columns through one-hot encodding will be a lot.
be = BinaryEncoder(cols=['rating'])
OTT_DF = be.fit_transform(OTT_DF)

# label encoding the values for platform: [manual label encodding]
label_encoder_platform = {'Disney': 0, 'Netflix' : 1}
OTT_DF['platform'] = (OTT_DF['platform'].replace(label_encoder_platform)).astype(int)

label_encoder_type = {'Movie': 0, 'TV Show' : 1}
OTT_DF['type'] = (OTT_DF['type'].replace(label_encoder_type)).astype(int)
OTT_DF.head()

,type,title,director,cast,country,rating_0,rating_1,rating_2,rating_3,listed_in,description,platform,duration_movies_bin,duration_tv_series_bin,date_added_bin,release_year_bin
id,,,,,,,,,,,,,,,,
1,0,ChuChuTV Surprise Eggs Learning Videos (English),Unknown,Unknown,Unknown,0,0,0,1,Children & Family Movies,From colors and letters to animals of all kind...,1,1,0,1,4
2,0,The Journey Is the Destination,Bronwen Hughes,"Ben Schnetzer, Kelly Macdonald, Sam Hazeldine,...",United States,0,0,1,0,Dramas,Spirited 22-year-old activist and photojournal...,1,4,0,0,2
3,1,Champions,Unknown,"Anders Holm, Fortune Feimster, Andy Favreau, J...",United States,0,0,1,1,TV Comedies,"Years after getting his girlfriend pregnant, w...",1,0,0,3,3
4,1,The Returned,Unknown,"Anne Cosigny, Frédéric Pierrot, Clotilde Hesme...",France,0,1,0,0,"International TV Shows, TV Dramas, TV Horror",On returning home and finding they're believed...,1,0,0,1,2
5,0,Super Bheem Bana Vajraveer,Sumit Das,"Sonal Kaushal, Rupa Bhimani, Julie Tejwani, Sa...",India,0,1,0,1,Children & Family Movies,"Hoping to find a magical root, a monster has c...",1,1,0,1,3


In [12]:
OTT_DF.info()

<class 'pandas.DataFrame'>
RangeIndex: 9338 entries, 1 to 9338
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   type                    9338 non-null   int64
 1   title                   9338 non-null   str  
 2   director                9338 non-null   str  
 3   cast                    9338 non-null   str  
 4   country                 9338 non-null   str  
 5   rating_0                9338 non-null   int64
 6   rating_1                9338 non-null   int64
 7   rating_2                9338 non-null   int64
 8   rating_3                9338 non-null   int64
 9   listed_in               9338 non-null   str  
 10  description             9338 non-null   str  
 11  platform                9338 non-null   int64
 12  duration_movies_bin     9338 non-null   int64
 13  duration_tv_series_bin  9338 non-null   int64
 14  date_added_bin          9338 non-null   int64
 15  release_year_bin        9338 non

In [13]:
def tfidf_svd_features(df, text_cols, variance_threshold=0.85):
    tfidf_matrices = []

    for col in text_cols:
        vectorizer = TfidfVectorizer(
            stop_words='english',
            max_features=1000
        )

        X = vectorizer.fit_transform(
            df[col].fillna('').astype(str)
        )

        tfidf_matrices.append(X)

    X_text = hstack(tfidf_matrices)

    max_components = min(X_text.shape) - 1

    svd = TruncatedSVD(
        n_components=max_components,
        random_state=42
    )

    X_svd = svd.fit_transform(X_text)

    cum_var = np.cumsum(svd.explained_variance_ratio_)
    n_components = np.argmax(cum_var >= variance_threshold) + 1

    X_svd = X_svd[:, :n_components]

    svd_df = pd.DataFrame(
        X_svd,
        columns=[f'text_svd_{i}' for i in range(n_components)],
        index=df.index
    )

    return svd_df

# -----------------------------------------------------------------
cat_cols = ['country', 'title', 'director', 'cast', 'description']

text_features = tfidf_svd_features(
    OTT_DF,
    cat_cols,
    variance_threshold=0.85
)

OTT_DF = pd.concat([OTT_DF, text_features], axis=1)
OTT_DF.drop(cat_cols, axis=1, inplace=True)

## 3. Modelling

In [50]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multiclass import OneVsRestClassifier

from sklearn.ensemble import BaggingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    f1_score,
    accuracy_score
)

from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from pprint import pprint

import warnings
warnings.filterwarnings(
    "ignore",
    category=UserWarning,
    module=r"sklearn\.multiclass",
    message=r"Label .* is present in all training examples\."
)

In [29]:
# Prepare training data

df = OTT_DF.copy()

# Convert genre strings to list of labels
y_raw = (
    df["listed_in"]
    .astype(str)
    .str.split(",")
    .apply(lambda x: [i.strip() for i in x])
)

# Multi-label encoding
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(y_raw)

# Features
X = df.drop(columns=["listed_in"])

print(f"Shape of X: {X.shape}")
print(f"Shape of Y: {Y.shape}")
print(f"Number of genres: {len(mlb.classes_)}")

Shape of X: (9338, 1704)
Shape of Y: (9338, 84)
Number of genres: 84


In [32]:
def intersection_accuracy(y_true, y_pred):
    """
    Computes average intersection accuracy:
    |Actual ∩ Predicted| / |Actual|

    Parameters
    ----------
    y_true : ndarray (n_samples, n_labels)
    y_pred : ndarray (n_samples, n_labels)

    Returns
    -------
    float
        Average intersection accuracy.
    """

    # Number of correctly predicted labels per sample
    intersection = np.logical_and(y_true, y_pred).sum(axis=1)

    # Number of actual labels per sample
    actual = y_true.sum(axis=1)

    # Avoid division by zero
    scores = np.divide(
        intersection,
        actual,
        out=np.zeros_like(intersection, dtype=float),
        where=actual != 0
    )

    return float(scores.mean())

# cross-valiation object for multilable classification

cv = MultilabelStratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)


In [ ]:
# -- Logistic Regression --
lr_model = OneVsRestClassifier(
    LogisticRegression(
        solver="liblinear",
        C=1.0,
        max_iter=750,
        class_weight="balanced",
        random_state=42
    ),
    n_jobs=-2
)

lr_scores = []

print("\nRunning Logistic Regression CV...\n")

for fold, (train_idx, test_idx) in enumerate(cv.split(X, Y), start=1):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    Y_train = Y[train_idx]
    Y_test = Y[test_idx]

    lr_model.fit(X_train, Y_train)

    Y_pred = lr_model.predict(X_test)

    metrics = {
        "Fold": fold,
        "Micro_F1": round(f1_score(Y_test, Y_pred, average="micro", zero_division=0), 6),
        "Macro_F1": round(f1_score(Y_test, Y_pred, average="macro", zero_division=0), 6),
        "Accuracy": round(accuracy_score(Y_test, Y_pred), 6),
        "Intersection_Accuracy": round(intersection_accuracy(Y_test, Y_pred), 6),
    }

    lr_scores.append(metrics)

    pprint(metrics)

lr_results = pd.DataFrame(lr_scores)

print("\nFold-wise Results:")
print(lr_results)

print("\nAverage Logistic Regression Metrics:")
print(lr_results.mean(numeric_only=True))


Running Logistic Regression CV...

{'Accuracy': 0.101381,
 'Fold': 1,
 'Hamming_Loss': 0.028963,
 'Intersection_Accuracy': 0.80087,
 'Macro_F1': 0.357494,
 'Micro_F1': 0.599595}
{'Accuracy': 0.094428,
 'Fold': 2,
 'Hamming_Loss': 0.029729,
 'Intersection_Accuracy': 0.807281,
 'Macro_F1': 0.356985,
 'Micro_F1': 0.590128}

Fold-wise Results:
   Fold  Micro_F1  Macro_F1  Accuracy  Intersection_Accuracy  Hamming_Loss
0     1  0.599595  0.357494  0.101381               0.800870      0.028963
1     2  0.590128  0.356985  0.094428               0.807281      0.029729

Average Logistic Regression Metrics:
Fold                     1.500000
Micro_F1                 0.594861
Macro_F1                 0.357239
Accuracy                 0.097905
Intersection_Accuracy    0.804075
Hamming_Loss             0.029346
dtype: float64


In [58]:
# -- SVC --
svc_model = OneVsRestClassifier(
    LinearSVC(
        C=1.0,
        class_weight="balanced",
        loss="squared_hinge",
        dual=False,
        max_iter=5000,
        random_state=42
    ),
    n_jobs=-2
)

svc_scores = []

print("\nRunning SVC CV...\n")

for fold, (train_idx, test_idx) in enumerate(cv.split(X, Y), start=1):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    Y_train = Y[train_idx]
    Y_test = Y[test_idx]

    svc_model.fit(X_train, Y_train)

    Y_pred = svc_model.predict(X_test)

    metrics = {
        "Fold": fold,
        "Micro_F1": round(f1_score(Y_test, Y_pred, average="micro", zero_division=0), 6),
        "Macro_F1": round(f1_score(Y_test, Y_pred, average="macro", zero_division=0), 6),
        "Accuracy": round(accuracy_score(Y_test, Y_pred), 6),
        "Intersection_Accuracy": round(intersection_accuracy(Y_test, Y_pred), 6),
    }

    svc_scores.append(metrics)

    pprint(metrics)

svc_results = pd.DataFrame(svc_scores)

print("\nFold-wise Results:")
print(svc_results)

print("\nAverage SVC Metrics:")
print(svc_results.mean(numeric_only=True))


Running SVC CV...

{'Accuracy': 0.195858,
 'Fold': 1,
 'Intersection_Accuracy': 0.647793,
 'Macro_F1': 0.356009,
 'Micro_F1': 0.623616}
{'Accuracy': 0.186942,
 'Fold': 2,
 'Intersection_Accuracy': 0.656919,
 'Macro_F1': 0.345888,
 'Micro_F1': 0.616166}

Fold-wise Results:
   Fold  Micro_F1  Macro_F1  Accuracy  Intersection_Accuracy
0     1  0.623616  0.356009  0.195858               0.647793
1     2  0.616166  0.345888  0.186942               0.656919

Average SVC Metrics:
Fold                     1.500000
Micro_F1                 0.619891
Macro_F1                 0.350948
Accuracy                 0.191400
Intersection_Accuracy    0.652356
dtype: float64


In [ ]:
# --- LightGBM ---

lgbm_model = OneVsRestClassifier(
    LGBMClassifier(
        objective="binary",
        boosting_type="gbdt",
        n_estimators=75,
        learning_rate=0.05,
        num_leaves=31,
        max_depth=8,
        min_child_samples=20,
        feature_fraction=0.8,
        bagging_fraction=0.8,
        bagging_freq=5,
        verbosity=-1,
        random_state=42
    )
)
lgbm_scores = []

print("\nRunning LightGBM CV...\n")

for fold, (train_idx, test_idx) in enumerate(cv.split(X, Y), start=1):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    Y_train = Y[train_idx]
    Y_test = Y[test_idx]

    lgbm_model.fit(X_train, Y_train)

    Y_pred = lgbm_model.predict(X_test)

    metrics = {
        "Fold": fold,
        "Micro_F1": round(f1_score(Y_test, Y_pred, average="micro", zero_division=0), 6),
        "Macro_F1": round(f1_score(Y_test, Y_pred, average="macro", zero_division=0), 6),
        "Accuracy": round(accuracy_score(Y_test, Y_pred), 6),
        "Intersection_Accuracy": round(intersection_accuracy(Y_test, Y_pred), 6),
    }
    
    lgbm_scores.append(metrics)

    pprint(metrics)

lgbm_results = pd.DataFrame(lgbm_scores)

print(lgbm_results)

print("\nAverage LightGBM Metrics")
print(lgbm_results.mean(numeric_only=True))



Running LightGBM CV...

{'Accuracy': 0.13503,
 'Fold': 1,
 'Hamming_Loss': 0.01937,
 'Intersection_Accuracy': 0.371441,
 'Macro_F1': 0.165388,
 'Micro_F1': 0.506639}
{'Accuracy': 0.147171,
 'Fold': 2,
 'Hamming_Loss': 0.018905,
 'Intersection_Accuracy': 0.381327,
 'Macro_F1': 0.165476,
 'Micro_F1': 0.513709}
   Fold  Micro_F1  Macro_F1  Accuracy  Intersection_Accuracy  Hamming_Loss
0     1  0.506639  0.165388  0.135030               0.371441      0.019370
1     2  0.513709  0.165476  0.147171               0.381327      0.018905

Average LightGBM Metrics
Fold                     1.500000
Micro_F1                 0.510174
Macro_F1                 0.165432
Accuracy                 0.141101
Intersection_Accuracy    0.376384
Hamming_Loss             0.019138
dtype: float64


In [ ]:
# --- Bagged LogisticRegression ---
# number off models = 50 [n_estimators] * 84 [classes] * 2 [CV] = 8,400
# number off models = 7 [n_estimators] * 84 [classes] * 2 [CV] = 1176
# approximate time  
base_lr = LogisticRegression(
    solver="liblinear",
    max_iter=750,
    class_weight="balanced"
)

bagged_lr_model = OneVsRestClassifier(
    BaggingClassifier(
        estimator=base_lr,
        n_estimators=14,
        max_samples=0.8,
        max_features=1.0,
        bootstrap=True,
        random_state=42,
        n_jobs=-2
    )
)

bagged_lr_scores = []

print("\nRunning Bagged Logistic Regression CV...\n")

for fold, (train_idx, test_idx) in enumerate(cv.split(X, Y), start=1):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    Y_train = Y[train_idx]
    Y_test = Y[test_idx]

    bagged_lr_model.fit(X_train, Y_train)

    Y_pred = bagged_lr_model.predict(X_test)

    metrics = {
        "Fold": fold,
        "Micro_F1": round(f1_score(Y_test, Y_pred, average="micro", zero_division=0), 6),
        "Macro_F1": round(f1_score(Y_test, Y_pred, average="macro", zero_division=0), 6),
        "Accuracy": round(accuracy_score(Y_test, Y_pred), 6),
        "Intersection_Accuracy": round(intersection_accuracy(Y_test, Y_pred), 6),
    }
    
    bagged_lr_scores.append(metrics)

    pprint(metrics)

bagged_lr_results = pd.DataFrame(bagged_lr_scores)

print(bagged_lr_results)

print("\nAverage Bagged Logistic Regression Metrics")
print(bagged_lr_results.mean(numeric_only=True))



Running Bagged Logistic Regression CV...

{'Accuracy': 0.00302,
 'Fold': 1,
 'Hamming_Loss': 0.047105,
 'Intersection_Accuracy': 0.75151,
 'Macro_F1': 0.355437,
 'Micro_F1': 0.463092}
{'Accuracy': 0.000213,
 'Fold': 2,
 'Hamming_Loss': 0.039861,
 'Intersection_Accuracy': 0.754324,
 'Macro_F1': 0.339636,
 'Micro_F1': 0.500159}
   Fold  Micro_F1  Macro_F1  Accuracy  Intersection_Accuracy  Hamming_Loss
0     1  0.463092  0.355437  0.003020               0.751510      0.047105
1     2  0.500159  0.339636  0.000213               0.754324      0.039861

Average Bagged Logistic Regression Metrics
Fold                     1.500000
Micro_F1                 0.481626
Macro_F1                 0.347537
Accuracy                 0.001617
Intersection_Accuracy    0.752917
Hamming_Loss             0.043483
dtype: float64


In [52]:
# --- Bagged SVC ---
 
base_svc = LinearSVC(
    C=1.0,
    class_weight="balanced"
)

bagged_svc_model = OneVsRestClassifier(
    BaggingClassifier(
        estimator=base_svc,
        n_estimators=5,
        max_samples=0.8,
        max_features=1.0,
        bootstrap=True,
        random_state=42,
        n_jobs=-2
    )
)

bagged_svc_scores = []

print("\nRunning Bagged SVC CV...\n")

for fold, (train_idx, test_idx) in enumerate(cv.split(X, Y), start=1):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    Y_train = Y[train_idx]
    Y_test = Y[test_idx]

    bagged_svc_model.fit(X_train, Y_train)

    Y_pred = bagged_svc_model.predict(X_test)

    metrics = {
        "Fold": fold,
        "Micro_F1": round(f1_score(Y_test, Y_pred, average="micro", zero_division=0), 6),
        "Macro_F1": round(f1_score(Y_test, Y_pred, average="macro", zero_division=0), 6),
        "Accuracy": round(accuracy_score(Y_test, Y_pred), 6),
        "Intersection_Accuracy": round(intersection_accuracy(Y_test, Y_pred), 6),
    }
    
    bagged_svc_scores.append(metrics)

    pprint(metrics)

bagged_svc_results = pd.DataFrame(bagged_svc_scores)

print(bagged_svc_results)

print("\nAverage Bagged SVC Metrics")
print(bagged_svc_results.mean(numeric_only=True))



Running Bagged SVC CV...

{'Accuracy': 0.007118,
 'Fold': 1,
 'Intersection_Accuracy': 0.595377,
 'Macro_F1': 0.297818,
 'Micro_F1': 0.500242}
{'Accuracy': 0.0,
 'Fold': 2,
 'Intersection_Accuracy': 0.604495,
 'Macro_F1': 0.295182,
 'Micro_F1': 0.427431}
   Fold  Micro_F1  Macro_F1  Accuracy  Intersection_Accuracy
0     1  0.500242  0.297818  0.007118               0.595377
1     2  0.427431  0.295182  0.000000               0.604495

Average Bagged SVC Metrics
Fold                     1.500000
Micro_F1                 0.463836
Macro_F1                 0.296500
Accuracy                 0.003559
Intersection_Accuracy    0.599936
dtype: float64


In [59]:
# Final Comparison

comparison = pd.DataFrame({
    "LogisticRegression": lr_results.mean(numeric_only=True),
    "SVC": svc_results.mean(numeric_only=True),
    "LightGBM": lgbm_results.mean(numeric_only=True),
    "Bagged_LogisticRegression": bagged_lr_results.mean(numeric_only=True),
    "Bagged_SVC": bagged_svc_results.mean(numeric_only=True)
}).T

display(comparison.sort_values("Intersection_Accuracy", ascending=False))

,Accuracy,Fold,Hamming_Loss,Intersection_Accuracy,Macro_F1,Micro_F1
LogisticRegression,0.097905,1.5,0.029346,0.804075,0.357239,0.594861
Bagged_LogisticRegression,0.001617,1.5,0.043483,0.752917,0.347537,0.481626
SVC,0.191400,1.5,NaN,0.652356,0.350948,0.619891
Bagged_SVC,0.003559,1.5,NaN,0.599936,0.296500,0.463836
LightGBM,0.141101,1.5,0.019138,0.376384,0.165432,0.510174
